# Auto Loader

## Objective

Implement incremental file ingestion using Databricks Auto Loader.

### Business Scenario

New NYC Taxi trip files arrive continuously in a landing area.

The ingestion framework must:

- Detect new files automatically
- Process only new files
- Handle schema changes
- Support late-arriving files
- Maintain ingestion state using checkpoints

### Output

- workspace.default.bronze_trip_autoloader

## Create Landing Zone

Auto Loader monitors a landing directory and discovers newly arrived files incrementally.

In [0]:
landing_path = (
    "/Volumes/workspace/default/"
    "raw_data/autoloader_trips"
)

checkpoint_path = (
    "/Volumes/workspace/default/"
    "raw_data/checkpoints/autoloader_trips"
)

print(landing_path)
print(checkpoint_path)

## Review Landing Files

In [0]:
display(
    dbutils.fs.ls(
        landing_path
    )
)

## Configure Auto Loader

Unlike spark.read.parquet(), Auto Loader tracks previously processed files and automatically identifies new arrivals.

In [0]:
streaming_trips = (
    spark.readStream
         .format("cloudFiles")
         .option(
             "cloudFiles.format",
             "parquet"
         )
         .option(
             "cloudFiles.schemaLocation",
             checkpoint_path
         )
         .load(
             landing_path
         )
)

## Review Inferred Schema

Auto Loader automatically infers schema from incoming files.

In [0]:
streaming_trips.printSchema()

## Create Bronze Auto Loader Table

In [0]:
(
    streaming_trips.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        checkpoint_path
    )
    .option(
        "mergeSchema",
        "true"
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        "workspace.default.bronze_trip_autoloader"
    )
)

## Validate Initial Load

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.default.bronze_trip_autoloader

## Incremental File Discovery

Copy another trip parquet file into the landing folder and rerun the Auto Loader write.

Expected Result:

- Previously processed files are skipped
- Only the newly arrived file is ingested


In [0]:
%sql
SELECT COUNT(*)
FROM workspace.default.bronze_trip_autoloader

## Late Arriving File Demonstration

Load an older trip file after newer files have already been processed.

Example:

- January file processed
- February file processed
- December file arrives later

Auto Loader processes the file because it is new to the landing directory, regardless of its event date.

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.default.bronze_trip_autoloader

## Demonstrate Schema Evolution

Create a new file containing an additional column such as:

promotion_code

Auto Loader can automatically evolve the schema when configured appropriately.

In [0]:
streaming_trips_schema_evolution = (
    spark.readStream
         .format("cloudFiles")
         .option(
             "cloudFiles.format",
             "parquet"
         )
         .option(
             "cloudFiles.schemaEvolutionMode",
             "addNewColumns"
         )
         .load(
             landing_path ="/Volumes/workspace/default/"
    "raw_data/autoloader_trips/*"
         )
)

## Validate Schema Evolution

In [0]:
spark.table(
    "workspace.default.bronze_trip_autoloader"
).printSchema()

## Small Files Discussion

Large numbers of very small files can create:

- Metadata overhead
- Additional Spark task scheduling
- Slower ingestion performance

Auto Loader efficiently tracks file discovery using metadata rather than repeatedly scanning the entire directory structure.

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.default.bronze_trip_autoloader

In [0]:
%sql
SELECT *
FROM workspace.default.bronze_trip_autoloader
LIMIT 20